# 🧠 The Imagination Stage Analysis (E0 → E1 → E2)

This notebook inspects the outputs of the Imagination pipeline:

| Stage | Operation | Artifacts |
|---|---|---|
| **E0** | Canonical camera setup & 2D silhouette rendering | `F/image.png`, `A/image.png`, `camera.json` |
| **E1** | Stable Diffusion completion (raw render, depth seeds, oracle) | `image.png`, selection score |
| **E2** | InstantMesh 3D "ghost" reconstruction & ICP alignment to anchor | `shape.npz` (raw & aligned), `mesh.obj`, `alignment.json` |

---

In [ ]:
import subprocess, sys, os
# Ensure package path is accessible
for p in ["/home/kpandey/satellite/pt_reg", "/home/kpandey/generative_assembly"]:
    if os.path.exists(p) and p not in sys.path:
        sys.path.insert(0, p)

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "plotly", "pandas", "numpy", "scipy"], check=True)
print("✅ Environment ready")

In [ ]:
import os, json
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from scipy.spatial import cKDTree
from IPython.display import display, Image, HTML
import warnings
warnings.filterwarnings('ignore')

GA_ROOT       = Path("/home/kpandey/generative_assembly")
DATASET_DIR   = GA_ROOT / "bottles498-inputs-v1"
DATASET_PATH  = DATASET_DIR / "dataset_2parts.json"
RUN_DIR       = GA_ROOT / "runs/imagination-study"

print(f"Dataset  : {'✅' if DATASET_PATH.exists() else '❌'}  {DATASET_PATH}")
print(f"Run dir  : {'✅' if RUN_DIR.exists() else '❌'}  {RUN_DIR}")

# Load dataset records
dataset_json = json.loads(DATASET_PATH.read_text())
cases = dataset_json.get("cases", dataset_json) if isinstance(dataset_json, dict) else dataset_json
print(f"Cases in study: {len(cases)}")

---
## 🔍 Index All Jobs Discovered in `RUN_DIR`

The pipeline stores jobs under `runs/imagination-study/jobs/<job_id>/result.json`.

In [ ]:
jobs_index = []
jobs_dir = RUN_DIR / "jobs"

if jobs_dir.exists():
    for res_path in sorted(jobs_dir.glob("*/result.json")):
        try:
            rec = json.loads(res_path.read_text())
            rec["dir"] = res_path.parent
            jobs_index.append(rec)
        except Exception:
            pass

print(f"✅ Total jobs found: {len(jobs_index)}")

df_jobs = pd.DataFrame([{
    "stage": j.get("stage"),
    "arm": j.get("arm"),
    "case_id": j.get("case_id"),
    "status": j.get("status"),
} for j in jobs_index])

if not df_jobs.empty:
    display(df_jobs.groupby(["stage", "status"]).size().to_frame("count"))
else:
    print("⚠️ No completed jobs found in RUN_DIR/jobs!")

def find_jobs(case_id, stage=None):
    """Find completed jobs for a given case and stage."""
    return [
        j for j in jobs_index
        if j.get("case_id") == case_id 
        and (stage is None or j.get("stage") == stage)
        and j.get("status") == "complete"
    ]

---
## 📦 Load Ground Truth Case

In [ ]:
# Try importing data loader from generative_assembly
try:
    from generative_assembly import data
    has_ga_pkg = True
except ImportError:
    has_ga_pkg = False

# Load study config
cfg_path = RUN_DIR / "study.json"
if cfg_path.exists():
    cfg = json.loads(cfg_path.read_text()).get("identity", {}).get("config", {})
else:
    cfg = {}

def load_gt(case_rec):
    """
    Loads canonical normalized fragments and ground-truth complete object.
    """
    cid = case_rec["id"]
    
    # 1. Preferred: use package to load canonical points + reference complete
    if has_ga_pkg and cfg:
        try:
            c_obj = data.load_case(DATASET_PATH, case_rec, cfg)
            r_obj = data.reference(DATASET_PATH, c_obj)
            comp = r_obj["complete"] if r_obj and "complete" in r_obj else None
            return c_obj["points"], comp
        except Exception as e:
            pass

    # 2. Fallback: load from E0 job observed.npz and evaluator_only
    e0_jobs = find_jobs(cid, "E0")
    frags, comp = [], None
    if e0_jobs:
        obs_file = e0_jobs[0]["dir"] / "observed.npz"
        if obs_file.exists():
            with np.load(obs_file, allow_pickle=False) as f:
                p_keys = sorted([k for k in f.files if k.startswith("points_")])
                frags = [f[k] for k in p_keys]
    
    eval_idx = DATASET_DIR / "evaluator_only" / "index.json"
    if eval_idx.exists():
        idx_map = json.loads(eval_idx.read_text())
        if cid in idx_map:
            ref_path = DATASET_DIR / idx_map[cid]["path"]
            if ref_path.exists():
                with np.load(ref_path, allow_pickle=False) as f:
                    if "complete" in f.files:
                        comp = f["complete"]
    return frags, comp

print("✅ GT loader ready")

---
## 🗂️ Select Case to Inspect

In [ ]:
CASE_IDX = 0  # ← Change index (0..9) to view different cases

case_rec = cases[CASE_IDX]
CASE_ID  = case_rec["id"]

print(f"Case ID    : {CASE_ID}")
print(f"Pieces     : {case_rec.get('pieces', 2)}")
print(f"Category   : {case_rec.get('category')}")
print(f"Difficulty : {case_rec.get('difficulty_band')}")
print(f"Cut family : {case_rec.get('cut_family')}")

fragments_gt, complete_gt = load_gt(case_rec)
print(f"\nGT fragments: {len(fragments_gt)} pieces")
for i, p in enumerate(fragments_gt):
    print(f"  Fragment {i}: {len(p)} points")
print(f"GT complete shape: {'✅ Present (' + str(len(complete_gt)) + ' points)' if complete_gt is not None else '❌ Not available'}")

---
## 📸 Stage E0 — Rendered Input Images

- **`F`**: Anchor fragment only (the prompt fragment given to the AI).
- **`A`**: All fragments placed together.

In [ ]:
e0_jobs = find_jobs(CASE_ID, "E0")

if not e0_jobs:
    print(f"❌ No E0 run found for {CASE_ID}")
else:
    e0_dir = e0_jobs[0]["dir"]
    
    images = {}
    for kind in ["F", "A"]:
        img_path = e0_dir / kind / "image.png"
        if img_path.exists():
            images[kind] = img_path

    if images:
        html = "<div style='display: flex; gap: 24px; align-items: flex-start;'>"
        for kind, img_path in sorted(images.items()):
            title = "Anchor Fragment Only (F)" if kind == "F" else "All Fragments (A)"
            html += f"""
            <div style='text-align: center'>
              <p style='font-weight:bold; color:#eee'>{title}</p>
              <img src='{img_path}' width='280' style='border:1px solid #444; border-radius:6px'/>
            </div>"""
        html += "</div>"
        display(HTML(html))
    
    cam_file = e0_dir / "camera.json"
    if cam_file.exists():
        cam = json.loads(cam_file.read_text())
        print(f"Camera Extent: {cam.get('extent')}, Pixels: {cam.get('pixels')}")

---
## 🎨 Stage E1 — Stable Diffusion Completions

Compares all 2D generation attempts across seeds and input types:

In [ ]:
e1_jobs = find_jobs(CASE_ID, "E1") + find_jobs(CASE_ID, "E1_ORACLE")

if not e1_jobs:
    print(f"❌ No E1 jobs found for {CASE_ID}")
else:
    print(f"Found {len(e1_jobs)} E1 generated variations:")
    
    html = "<div style='display: flex; flex-wrap: wrap; gap: 14px;'>"
    for j in sorted(e1_jobs, key=lambda x: x.get("arm", "")):
        img_path = j["dir"] / "image.png"
        if img_path.exists():
            arm = j.get("arm", "")
            out_meta = j.get("output", {})
            score = out_meta.get("selection", {}).get("selection_score")
            score_str = f"score={score:.3f}" if score is not None else ""
            
            if "true_image" in arm:
                color = "#2ecc71"
                badge = "ORACLE (True Image)"
            elif "raw" in arm:
                color = "#3498db"
                badge = "Raw Silhouette"
            elif "sd15_depth" in arm:
                color = "#e67e22"
                badge = "SD 1.5 + Depth"
            elif "sd15" in arm:
                color = "#9b59b6"
                badge = "SD 1.5 Pure (No Depth)"
            elif "gemini" in arm:
                color = "#1abc9c"
                badge = "Gemini API"
            elif "sdxl" in arm:
                color = "#f1c40f"
                badge = "SDXL Inpaint"
            else:
                color = "#95a5a6"
                badge = arm.split("__")[0]

            html += f"""
            <div style='text-align:center; border:1px solid #333; padding:8px; border-radius:8px; background:#161b22'>
              <span style='font-size:10px; font-weight:bold; color:{color}; text-transform:uppercase'>{badge}</span>
              <p style='font-size:11px; color:#aaa; margin:2px 0'>{arm}</p>
              <p style='font-size:10px; color:#888; margin:0 0 4px 0'>{score_str}</p>
              <img src='{img_path}' width='180' style='border-radius:4px'/>
            </div>"""
    html += "</div>"
    display(HTML(html))

---
## 🧊 Stage E2 — InstantMesh 3D "Ghost" Reconstruction vs Ground Truth

Calculates **Bidirectional Chamfer Distance**, **Precision@2cm**, and **Recall@2cm** between the InstantMesh 3D point cloud and the Ground Truth bottle.

In [ ]:
e2_jobs = find_jobs(CASE_ID, "E2")
e2_rows = []

if not e2_jobs:
    print(f"❌ No E2 jobs found for {CASE_ID}")
else:
    print(f"Found {len(e2_jobs)} E2 reconstructions")
    
    for j in e2_jobs:
        shape_file = j["dir"] / "shape.npz"
        align_file = j["dir"] / "alignment.json"
        arm = j.get("arm", j["dir"].name)
        
        if not shape_file.exists():
            continue
            
        with np.load(shape_file) as f:
            algn_pts = f["aligned"]
        
        row = {"Arm": arm, "Points": len(algn_pts)}
        
        if complete_gt is not None:
            tree_gt = cKDTree(complete_gt)
            tree_im = cKDTree(algn_pts)
            d_im_to_gt = tree_gt.query(algn_pts)[0]
            d_gt_to_im = tree_im.query(complete_gt)[0]
            row["Chamfer Dist (↓)"] = float((d_im_to_gt.mean() + d_gt_to_im.mean()) / 2)
            row["Precision@2cm (↑)"] = float((d_im_to_gt < 0.02).mean())
            row["Recall@2cm (↑)"] = float((d_gt_to_im < 0.02).mean())
            row["F1@2cm (↑)"] = float(2 * row["Precision@2cm (↑)"] * row["Recall@2cm (↑)"] / max(row["Precision@2cm (↑)"] + row["Recall@2cm (↑)"], 1e-6))
            
        if align_file.exists():
            fit = json.loads(align_file.read_text())
            row["ICP Fit Error"] = fit.get("fit_error")
            row["Held-out Error"] = fit.get("heldout_error")
            
        e2_rows.append(row)

if e2_rows:
    df_e2 = pd.DataFrame(e2_rows)
    display(df_e2.sort_values("Chamfer Dist (↓)" if "Chamfer Dist (↓)" in df_e2 else "Arm").round(4))

In [ ]:
# ── Interactive 3D Plot: GT Bottle vs InstantMesh Reconstructions ──────────────
fig = go.Figure()

# 1. Plot Ground Truth Fragments in distinct solid colors
FRAG_COLORS = ["#3498db", "#2ecc71", "#e74c3c", "#9b59b6"]
for i, p in enumerate(fragments_gt):
    fig.add_trace(go.Scatter3d(
        x=p[:, 0], y=p[:, 1], z=p[:, 2],
        mode="markers",
        marker=dict(size=2.5, color=FRAG_COLORS[i % len(FRAG_COLORS)], opacity=0.9),
        name=f"GT Fragment {i+1}"
    ))

# 2. Plot Reconstructed Shapes
ARM_PALETTE = px.colors.qualitative.Plotly
for idx, j in enumerate(e2_jobs):
    shape_file = j["dir"] / "shape.npz"
    if not shape_file.exists():
        continue
    with np.load(shape_file) as f:
        pts = f["aligned"]
    arm = j.get("arm", j["dir"].name)
    
    # Subsample for smooth 3D display if large
    if len(pts) > 2000:
        pts_show = pts[np.random.choice(len(pts), 2000, replace=False)]
    else:
        pts_show = pts
        
    is_oracle = "true_image" in arm
    is_raw = "raw" in arm
    
    clr = "#e74c3c" if is_oracle else ("#ffffff" if is_raw else ARM_PALETTE[idx % len(ARM_PALETTE)])
    
    fig.add_trace(go.Scatter3d(
        x=pts_show[:, 0], y=pts_show[:, 1], z=pts_show[:, 2],
        mode="markers",
        marker=dict(size=1.8, color=clr, opacity=0.45),
        name=f"InstantMesh: {arm}",
        visible="legendonly" if (idx > 1 and not is_oracle) else True
    ))

fig.update_layout(
    title=f"3D Reconstruction: Ground Truth Fragments vs InstantMesh Shape ({CASE_ID})<br>" \
          "<sup>Solid blue/green = GT fragments | Semitransparent points = InstantMesh hallucinations</sup>",
    scene=dict(
        xaxis_title="X", yaxis_title="Y", zaxis_title="Z",
        bgcolor="#0d1117",
        aspectmode="data",
    ),
    template="plotly_dark", height=750,
    legend=dict(itemsizing="constant"),
)
fig.show()

---
## 📊 Aggregate Metrics Across All 10 Cases

Quantifies overall performance across the entire 10-bottle dataset:

In [ ]:
study_metrics = []

for c_rec in cases:
    cid = c_rec["id"]
    frags, comp = load_gt(c_rec)
    if comp is None:
        continue
        
    tree_gt = cKDTree(comp)
    
    for j in find_jobs(cid, "E2"):
        shape_file = j["dir"] / "shape.npz"
        align_file = j["dir"] / "alignment.json"
        if not shape_file.exists():
            continue
            
        with np.load(shape_file) as f:
            algn_pts = f["aligned"]
            
        tree_im = cKDTree(algn_pts)
        d_im_gt = tree_gt.query(algn_pts)[0]
        d_gt_im = tree_im.query(comp)[0]
        
        chamfer = float((d_im_gt.mean() + d_gt_im.mean()) / 2)
        prec = float((d_im_gt < 0.02).mean())
        rec  = float((d_gt_im < 0.02).mean())
        f1   = float(2 * prec * rec / max(prec + rec, 1e-6))
        
        arm = j.get("arm", j["dir"].name)
        if "true_image" in arm:
            category_type = "Oracle Image"
        elif "raw" in arm:
            category_type = "Raw Silhouette"
        elif "sd15_depth" in arm:
            category_type = "SD 1.5 + Depth"
        elif "sd15" in arm:
            category_type = "SD 1.5 Pure (No Depth)"
        elif "gemini" in arm:
            category_type = "Gemini API"
        elif "sdxl" in arm:
            category_type = "SDXL Inpaint"
        else:
            category_type = arm.split("__")[0]
        
        row = {
            "case_id": cid,
            "difficulty": c_rec.get("difficulty_band", "unknown"),
            "cut_family": c_rec.get("cut_family", "unknown"),
            "arm": arm,
            "type": category_type,
            "chamfer": chamfer,
            "precision_2cm": prec,
            "recall_2cm": rec,
            "f1_2cm": f1,
        }
        if align_file.exists():
            fit = json.loads(align_file.read_text())
            row["fit_error"] = fit.get("fit_error")
            row["heldout_error"] = fit.get("heldout_error")
            
        study_metrics.append(row)

df_study = pd.DataFrame(study_metrics)

if not df_study.empty:
    print(f"Evaluated {len(df_study)} total 3D reconstructions across all cases.")
    print("\n📋 Average Quality by Image Source:")
    display(df_study.groupby("type")[["chamfer", "precision_2cm", "recall_2cm", "f1_2cm"]].mean().round(4))
    
    # Box plot comparing reconstruction errors
    fig = px.box(
        df_study,
        x="type", y="chamfer",
        color="type",
        points="all",
        labels={"chamfer": "Chamfer Distance (lower is better)", "type": "Image Source"},
        title="InstantMesh 3D Accuracy: Raw Silhouette vs SD1.5 vs Oracle",
        template="plotly_dark", height=480
    )
    fig.show()
else:
    print("⚠️ No metric rows generated. Check if evaluator_only references exist.")